# Module 09: Advanced Matching


## 🚀 Introduction to PhraseMatcher

In Module 8, we used the `Matcher` to define token-by-token dictionary rules. This is powerful, but if you have a massive dictionary of 10,000 exact phrases (like a list of medical drugs or country names), writing 10,000 dictionary rules is slow and tedious.

The **`PhraseMatcher`** lets you match large lists of strings very efficiently by passing in `Doc` objects as patterns!


In [1]:
import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.load("en_core_web_sm")
matcher = PhraseMatcher(nlp.vocab)

# A list of exact phrases we want to find
terms = ["Barack Obama", "Angela Merkel", "Washington, D.C."]

# We must convert the string terms into Doc objects!
patterns = [nlp.make_doc(text) for text in terms]

# Add the patterns to the PhraseMatcher
matcher.add("Politicians", patterns)

doc = nlp("Angela Merkel met with Barack Obama in Washington, D.C. yesterday.")

print("Found Matches:")
for match_id, start, end in matcher(doc):
    print(f"- {doc[start:end].text}")


Found Matches:
- Angela Merkel
- Barack Obama
- Washington, D.C.


<br><br>

---

<br><br>


## 🔠 Matching on Different Attributes

By default, the `PhraseMatcher` looks for exact string matches (`ORTH`). 
If your list is "Barack Obama", it won't match "barack obama" (lowercase).

You can tell the PhraseMatcher to compare the `LOWER` attribute or the `LEMMA` attribute instead!


In [2]:
# Re-initialize the matcher, telling it to compare lowercase text
matcher_lower = PhraseMatcher(nlp.vocab, attr="LOWER")

# Add the patterns again
matcher_lower.add("Politicians", patterns)

doc2 = nlp("angela merkel met with barack obama in washington, d.c.")

print("Found Matches (Case Insensitive):")
for match_id, start, end in matcher_lower(doc2):
    print(f"- {doc2[start:end].text}")


Found Matches (Case Insensitive):
- angela merkel
- barack obama


<br><br>

---

<br><br>


## 🌳 The DependencyMatcher

The standard `Matcher` looks at tokens sequentially (Token 1, then Token 2).
But language is flexible. "The dog chased the cat" vs "The cat was chased by the dog". The exact tokens are in a different order, but the grammatical relationship is the same!

The **`DependencyMatcher`** lets you find patterns in the *Dependency Tree* (which we covered in Module 5), rather than the sequential text.


In [3]:
from spacy.matcher import DependencyMatcher

dep_matcher = DependencyMatcher(nlp.vocab)

# We want to find cases where a dog is chasing a cat, regardless of word order.
# Rule 1: Find the verb 'chase'
# Rule 2: Find 'dog' acting as the subject of 'chase'
# Rule 3: Find 'cat' acting as the object of 'chase'

pattern = [
    {"RIGHT_ID": "chase_verb", "RIGHT_ATTRS": {"LEMMA": "chase", "POS": "VERB"}}, 
    # 'dog' must be the subject (nsubj) of 'chase'
    {"LEFT_ID": "chase_verb", "REL_OP": ">", "RIGHT_ID": "dog_subj", "RIGHT_ATTRS": {"LEMMA": "dog", "DEP": "nsubj"}},
    # 'cat' must be the direct object (dobj) of 'chase'
    {"LEFT_ID": "chase_verb", "REL_OP": ">", "RIGHT_ID": "cat_obj", "RIGHT_ATTRS": {"LEMMA": "cat", "DEP": "dobj"}}
]

dep_matcher.add("DogChaseCat", [pattern])

doc1 = nlp("The clever dog quickly chased the red cat.")
doc2 = nlp("I saw a dog that loves to chase every cat it sees.")

# The matcher returns a list of tuples: (match_id, [token_indices_in_pattern_order])
print("Doc 1 Matches:", dep_matcher(doc1))
print("Doc 2 Matches:", dep_matcher(doc2))


Doc 1 Matches: [(12249515434332330393, [4, 2, 7])]
Doc 2 Matches: []


<br><br>

---

<br><br>


## 📏 The EntityRuler

The `EntityRuler` is arguably the most useful component in spaCy. 
It allows you to define `Matcher` or `PhraseMatcher` patterns, and automatically adds the matches to `doc.ents` (Named Entities).

This lets you combine statistical NER with hard-coded rules!


In [4]:
import spacy

# Start with a fresh model
nlp = spacy.load("en_core_web_sm")

# Let's see what the default model thinks "Apple" is here:
doc = nlp("Apple is looking to buy a U.K. startup.")
print(f"Default NER for Apple: {doc[0].ent_type_}")

# Add the EntityRuler to the pipeline BEFORE the statistical NER
# This tells spaCy: "If my hard-coded rule matches, trust me over the AI"
ruler = nlp.add_pipe("entity_ruler", before="ner")

# We add a pattern to classify "Apple" as a FRUIT
patterns = [
    {"label": "FRUIT", "pattern": "Apple"}
]
ruler.add_patterns(patterns)

# Process the text again
doc_new = nlp("Apple is looking to buy a U.K. startup.")

print(f"New NER for Apple: {doc_new[0].ent_type_}")

# Notice the U.K. startup is still correctly identified by the statistical model!
print("All entities:", [(ent.text, ent.label_) for ent in doc_new.ents])


Default NER for Apple: ORG
New NER for Apple: FRUIT
All entities: [('Apple', 'FRUIT'), ('U.K.', 'GPE')]


<br><br>

---

<br><br>


## 🎉 Summary of Module 9

You've now mastered advanced pattern matching!
- You can use the `PhraseMatcher` to quickly scan for thousands of exact phrases using `LOWER` or `LEMMA`.
- You can navigate complex grammatical structures regardless of word order using the `DependencyMatcher`.
- You can inject rule-based logic directly into the statistical Named Entity Recognizer using the `EntityRuler`.

This marks the end of **Part 3: Pattern Matching & Rules**.

In **Module 10**, we will begin **Part 4: Customization & Extension**, learning how to write our own custom Python components and inject them into the spaCy pipeline!
